# RE:SOURCE AI · Task 2


## Deliberately break a Random Forest experiment

**Today:** Create one controlled failure so our future Diagnosis Agent has a real broken project to inspect.

We will keep the **Day 1 scientific reference** unchanged. If you are running in a fresh Colab session, this notebook will recreate the reference using the **same configuration as Day 1**. A fresh software environment may produce slightly different model results; that is why environment information must be recorded.

**Research focus**
- A script that runs is not necessarily a scientifically reproduced experiment.
- Repairs must preserve the original dataset, split, preprocessing, and model settings.

> This notebook contains only a small, known example written by us. Do not run unknown repositories or untrusted scripts in your main Colab environment.

### Step 1 · Import tools
`Path` creates folders; `subprocess` runs our own scripts and captures errors; `json` records a machine-readable test case. There is no autonomous LLM agent in Task 2 yet.

In [1]:
import json
import hashlib
import platform
import subprocess
import sys
import textwrap
import zipfile
from pathlib import Path

import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

ROOT = Path('/content/RE_SOURCE_AI') if Path('/content').is_dir() else Path.cwd() / 'RE_SOURCE_AI'
REF = ROOT / 'reference'
EXPERIMENTS = ROOT / 'experiments'
LOGS = ROOT / 'logs'
REPORTS = ROOT / 'reports'
for folder in [REF, EXPERIMENTS, LOGS, REPORTS]:
    folder.mkdir(parents=True, exist_ok=True)

def canonical_bytes(value):
    return json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False).encode('utf-8')

def sha256(data):
    return hashlib.sha256(data).hexdigest()

def write_json(path, value):
    path.write_text(json.dumps(value, indent=2, sort_keys=True) + '\n', encoding='utf-8')

print('Project folder:', ROOT)
print('scikit-learn version:', sklearn.__version__)

Project folder: /content/RE_SOURCE_AI
scikit-learn version: 1.6.1


### Step 2 · Preserve the original scientific settings
The configuration is **identical to Day 1**: Iris dataset, no preprocessing, stratified 80/20 split with seed 42, and a Random Forest with 100 trees and maximum depth 4.

If you already ran Day 1 in the same Colab session, this uses the existing configuration. It **never silently replaces a different reference**.

In [2]:
DAY1_CONFIG = {
    'experiment_id': 'iris_random_forest_v1',
    'dataset': {'name': 'Iris', 'source': 'sklearn.datasets.load_iris',
                'expected_samples': 150, 'features': 4, 'classes': 3},
    'preprocessing': {'steps': [], 'description': 'Raw numeric features; no scaling or imputation'},
    'split': {'test_size': 0.20, 'random_state': 42, 'stratify': True},
    'model': {'name': 'RandomForestClassifier',
              'parameters': {'n_estimators': 100, 'max_depth': 4,
                             'random_state': 42, 'n_jobs': 1}},
    'evaluation': {'metrics': ['accuracy', 'macro_f1'], 'labels': [0, 1, 2]},
}
config_path = REF / 'experiment_config.json'
if config_path.exists():
    ORIGINAL_CONFIG = json.loads(config_path.read_text(encoding='utf-8'))
    assert ORIGINAL_CONFIG == DAY1_CONFIG, 'Found a different reference configuration. Do not overwrite it.'
    print('Existing Day 1 configuration reused.')
else:
    ORIGINAL_CONFIG = DAY1_CONFIG
    write_json(config_path, ORIGINAL_CONFIG)
    print('Day 1 configuration recreated because this is a new runtime.')

METHOD_SHA = sha256(canonical_bytes(ORIGINAL_CONFIG))
print('Protected methodology SHA-256:', METHOD_SHA)

Day 1 configuration recreated because this is a new runtime.
Protected methodology SHA-256: 3d853b0c640c21a485954b317b8f65e3027fd8270eb95f8a3e48bfca8817a4de


### Step 3 · Record original baseline and exact train/test indices
If no Day 1 reference files exist in this session, we build them using the same settings. We keep saved split indices whenever available so the test data does not change without being noticed.

Accuracy is **measured here**, not assumed to be 96.67% on every computer.

In [3]:
iris = load_iris()
X = np.asarray(iris.data, dtype=np.float64)
y = np.asarray(iris.target, dtype=np.int64)
assert X.shape == (150, 4) and y.shape == (150,)

split_path = REF / 'split_indices.npz'
if split_path.exists():
    with np.load(split_path, allow_pickle=False) as split_file:
        train_idx = np.asarray(split_file['train_idx'], dtype=np.int64)
        test_idx = np.asarray(split_file['test_idx'], dtype=np.int64)
else:
    train_idx, test_idx = train_test_split(
        np.arange(len(y), dtype=np.int64),
        test_size=ORIGINAL_CONFIG['split']['test_size'],
        random_state=ORIGINAL_CONFIG['split']['random_state'], stratify=y
    )
    np.savez_compressed(split_path, train_idx=train_idx, test_idx=test_idx)

assert len(train_idx) == 120 and len(test_idx) == 30
assert len(set(train_idx).intersection(set(test_idx))) == 0
assert set(np.r_[train_idx, test_idx]) == set(range(len(y)))

model = RandomForestClassifier(**ORIGINAL_CONFIG['model']['parameters'])
model.fit(X[train_idx], y[train_idx])
pred = model.predict(X[test_idx])
measured_accuracy = float(accuracy_score(y[test_idx], pred))
measured_macro_f1 = float(f1_score(y[test_idx], pred, average='macro'))

DATA_SHA = sha256(
    canonical_bytes({'shape_X': list(X.shape), 'shape_y': list(y.shape),
                     'features': list(iris.feature_names), 'target_names': list(iris.target_names)})
    + np.asarray(X, dtype='<f8').tobytes(order='C')
    + np.asarray(y, dtype='<i8').tobytes(order='C')
)
SPLIT_SHA = sha256(
    np.asarray(train_idx, dtype='<i8').tobytes(order='C')
    + np.asarray(test_idx, dtype='<i8').tobytes(order='C')
)

metrics_path = REF / 'original_metrics.json'
if metrics_path.exists():
    saved_metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    assert saved_metrics['methodology_sha256'] == METHOD_SHA, 'Methodology changed.'
    assert saved_metrics['dataset_sha256'] == DATA_SHA, 'Dataset differs from Day 1.'
    assert saved_metrics['split_indices_sha256'] == SPLIT_SHA, 'Train/test split differs.'
    print('Saved Day 1 metrics reused for comparison.')
else:
    saved_metrics = {
        'experiment_id': ORIGINAL_CONFIG['experiment_id'],
        'methodology_sha256': METHOD_SHA,
        'dataset_sha256': DATA_SHA,
        'split_indices_sha256': SPLIT_SHA,
        'accuracy': measured_accuracy,
        'macro_f1': measured_macro_f1,
        'test_samples': len(test_idx),
    }
    write_json(metrics_path, saved_metrics)
    print('Fresh reference metrics saved.')

print(f'Current environment accuracy: {measured_accuracy:.4f} ({measured_accuracy:.2%})')
print(f'Original saved accuracy:     {saved_metrics["accuracy"]:.4f}')
if abs(measured_accuracy - saved_metrics['accuracy']) > 1e-12:
    print('NOTE: Current result differs from saved baseline. Investigate environment/version differences.')

Fresh reference metrics saved.
Current environment accuracy: 0.9667 (96.67%)
Original saved accuracy:     0.9667


### Step 4 · Create **working.py** and **broken.py**
Both scripts read **the exact same reference settings and split**. We create a copy of the functioning script, then inject one typo into the copy:

- Correct: `n_estimators` (number of trees)
- Broken: `n_estimator` (a parameter name the Random Forest constructor does not recognize)

This is a **code-level problem**, not an intentional change to the scientific configuration. Our future agent should fix the typo without changing tree count, split, or dataset.

In [4]:
working_source = textwrap.dedent(r'''    import json
    from pathlib import Path
    import numpy as np
    from sklearn.datasets import load_iris
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, f1_score

    ROOT = Path(__file__).resolve().parents[1]
    REF = ROOT / 'reference'
    config = json.loads((REF / 'experiment_config.json').read_text(encoding='utf-8'))
    iris = load_iris()
    X = np.asarray(iris.data, dtype=np.float64)
    y = np.asarray(iris.target, dtype=np.int64)
    with np.load(REF / 'split_indices.npz', allow_pickle=False) as saved:
        train_idx = saved['train_idx']
        test_idx = saved['test_idx']
    params = dict(config['model']['parameters'])
    # MODEL_PARAMETER_SETUP
    model = RandomForestClassifier(**params)
    model.fit(X[train_idx], y[train_idx])
    predictions = model.predict(X[test_idx])
    output = {
        'status': 'executed',
        'accuracy': float(accuracy_score(y[test_idx], predictions)),
        'macro_f1': float(f1_score(y[test_idx], predictions, average='macro')),
        'effective_n_estimators': model.get_params()['n_estimators'],
        'effective_max_depth': model.get_params()['max_depth'],
        'training_samples': len(train_idx),
        'testing_samples': len(test_idx),
    }
    print(json.dumps(output, sort_keys=True))
''')

broken_source = working_source.replace(
    '# MODEL_PARAMETER_SETUP',
    '# INTENTIONAL BUG: n_estimator is not a recognized constructor argument.\n'
    'params["n_estimator"] = params.pop("n_estimators")'
)
assert broken_source != working_source
assert 'params["n_estimator"]' in broken_source

working_path = EXPERIMENTS / 'working.py'
broken_path = EXPERIMENTS / 'broken.py'
working_path.write_text(working_source, encoding='utf-8')
broken_path.write_text(broken_source, encoding='utf-8')

print('Saved:', working_path)
print('Saved:', broken_path)
print('Fault injection: n_estimators -> n_estimator, in broken.py only')

Saved: /content/RE_SOURCE_AI/experiments/working.py
Saved: /content/RE_SOURCE_AI/experiments/broken.py
Fault injection: n_estimators -> n_estimator, in broken.py only


### Step 5 · Run both scripts and **capture the failure**
The notebook should **not crash** when `broken.py` fails. Instead, it captures the process return code and traceback. This traceback becomes the first input for our upcoming Diagnosis Agent.

Both executions are limited to our small, locally generated scripts and have a timeout.

In [5]:
def run_local_script(path):
    return subprocess.run(
        [sys.executable, str(path)],
        cwd=str(ROOT),
        capture_output=True, text=True, timeout=40, check=False
    )

working_run = run_local_script(working_path)
broken_run = run_local_script(broken_path)

(LOGS / 'working_stdout.txt').write_text(working_run.stdout, encoding='utf-8')
(LOGS / 'working_stderr.txt').write_text(working_run.stderr, encoding='utf-8')
(LOGS / 'broken_stdout.txt').write_text(broken_run.stdout, encoding='utf-8')
(LOGS / 'broken_stderr.txt').write_text(broken_run.stderr, encoding='utf-8')

print('WORKING script exit code:', working_run.returncode)
print('BROKEN  script exit code:', broken_run.returncode)
print('\nLast lines of broken error log:')
print('\n'.join(broken_run.stderr.strip().splitlines()[-6:]))

assert working_run.returncode == 0, 'The original script must run before we can evaluate its broken copy.'
assert broken_run.returncode != 0, 'The broken script unexpectedly executed successfully.'
assert 'n_estimator' in broken_run.stderr, 'Our injected error was not found in the traceback.'
print('\nPASS: Controlled failure reproduced and logged.')

WORKING script exit code: 0
BROKEN  script exit code: 1

Last lines of broken error log:
Traceback (most recent call last):
  File "/content/RE_SOURCE_AI/experiments/broken.py", line 20, in <module>
    model = RandomForestClassifier(**params)
TypeError: RandomForestClassifier.__init__() got an unexpected keyword argument 'n_estimator'. Did you mean 'n_estimators'?

PASS: Controlled failure reproduced and logged.


### Step 6 · Make the first machine-readable **repair test case**
This JSON file is what we will eventually give our agent. It contains the project paths, observed failure, allowed repair scope, and protected scientific settings. It does **not** contain an AI-generated repair.

A key lesson from Gap 3: simply preserving a configuration file does **not** prove that the executed code respected it. Our verifier will also need to inspect effective settings and rerun the experiment after repair.

In [6]:
working_output = json.loads(working_run.stdout.strip().splitlines()[-1])
assert abs(working_output['accuracy'] - measured_accuracy) < 1e-12
assert working_output['effective_n_estimators'] == 100
assert working_output['effective_max_depth'] == 4
assert json.loads(config_path.read_text(encoding='utf-8')) == ORIGINAL_CONFIG

CASE = {
    'case_id': 'RF001_invalid_constructor_parameter',
    'experiment_id': ORIGINAL_CONFIG['experiment_id'],
    'fault_type': 'invalid_model_parameter_name',
    'fault_injected_into': 'experiments/broken.py',
    'observed_exit_code': broken_run.returncode,
    'observed_error_excerpt': '\n'.join(broken_run.stderr.strip().splitlines()[-5:]),
    'status': 'BROKEN_NOT_REPAIRED',
    'reference_paths': [
        'reference/experiment_config.json',
        'reference/original_metrics.json',
        'reference/split_indices.npz',
    ],
    'repair_permissions': {
        'editable_files': ['experiments/broken.py'],
        'protected_files': ['reference/experiment_config.json',
                            'reference/original_metrics.json',
                            'reference/split_indices.npz'],
    },
    'expected_checks_after_repair': {
        'executes_without_error': True,
        'dataset_sha256': DATA_SHA,
        'split_indices_sha256': SPLIT_SHA,
        'methodology_sha256': METHOD_SHA,
        'expected_n_estimators': 100,
        'expected_max_depth': 4,
        'accuracy_reference': saved_metrics['accuracy'],
        'compare_methodology_and_results': True,
    },
    'runtime_versions': {'python': platform.python_version(),
                         'scikit_learn': sklearn.__version__,
                         'numpy': np.__version__},
}
case_path = REPORTS / 'task2_failure_case.json'
write_json(case_path, CASE)

print('CASE:', CASE['case_id'])
print('STATUS:', CASE['status'])
print('Original reference preserved:', json.loads(config_path.read_text()) == ORIGINAL_CONFIG)
print('Saved failure log:', LOGS / 'broken_stderr.txt')
print('Saved test case:', case_path)

CASE: RF001_invalid_constructor_parameter
STATUS: BROKEN_NOT_REPAIRED
Original reference preserved: True
Saved failure log: /content/RE_SOURCE_AI/logs/broken_stderr.txt
Saved test case: /content/RE_SOURCE_AI/reports/task2_failure_case.json


### Step 7 · Package the project and save it
This ZIP includes the reference configuration, split indices, original metrics, the working and broken script, error logs, and the failure-case JSON.

In Colab, the **Files** panel is temporary: download the ZIP before the session ends. Set `DOWNLOAD_NOW=True` in the optional final cell to trigger a browser download.

In [7]:
zip_path = ROOT.parent / 'RE_SOURCE_AI_Task2_Outputs.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file_path in sorted(ROOT.rglob('*')):
        if file_path.is_file():
            bundle.write(file_path, arcname=file_path.relative_to(ROOT))

print('Package:', zip_path)
print('Included files:')
with zipfile.ZipFile(zip_path) as bundle:
    for name in bundle.namelist():
        print('  ', name)

DOWNLOAD_NOW = False
if DOWNLOAD_NOW:
    try:
        from google.colab import files
        files.download(str(zip_path))
    except ImportError:
        print('Download manually from:', zip_path)

Package: /content/RE_SOURCE_AI_Task2_Outputs.zip
Included files:
   experiments/broken.py
   experiments/working.py
   logs/broken_stderr.txt
   logs/broken_stdout.txt
   logs/working_stderr.txt
   logs/working_stdout.txt
   reference/experiment_config.json
   reference/original_metrics.json
   reference/split_indices.npz
   reports/task2_failure_case.json


## Task 2 completed

| Item | Expected result |
|---|---|
| `experiments/working.py` | Runs and reports the model accuracy |
| `experiments/broken.py` | Fails with a parameter-name `TypeError` |
| `logs/broken_stderr.txt` | Contains the traceback |
| `reference/experiment_config.json` | Still contains 100 trees, depth 4, seed 42 |
| `reports/task2_failure_case.json` | Records a broken case ready for the Diagnosis Agent |
